# Kouman IA — Notebook 2 : Entraînement CPT Relais 1h sur GPU AMD ROCm (`Llama 3.2 1B` • `60M` Entraînables)

Ce notebook reprend exactement la pipeline de pré-entraînement continu (**CPT**) validée sur Gemma 2 2B, optimisée pour **`meta-llama/Llama-3.2-1B`** avec **~60M paramètres entraînables** sur GPU AMD ROCm :

### Architecture & Dimensionnement Exact (`59,95M` paramètres entraînables)
- **Modèle de base** : `meta-llama/Llama-3.2-1B` (`1,24B` params, `16` couches, `hidden_size=2048`, `intermediate_size=8192`, vocabulaire `128 256`, `bfloat16` natif ROCm, attention **`sdpa`** FlashAttention — 2× plus rapide que Gemma 2 `eager`).
- **Paramètres entraînables (`59 951 104` ≈ `60,0 M`, soit `4,63 %`)** :
  - **`LoRA r=85, alpha=170` (`use_rslora=True`)** sur les 7 projections linéaires (`q/k/v/o/gate/up/down_proj`) : **`59 883 520`** paramètres (`lr = 2.0e-4`).
  - **`modules_to_save=["input_layernorm", "post_attention_layernorm", "norm"]`** (RMSNorm pour absorber le décalage de distribution Baoulé/Dioula) : **`67 584`** paramètres (`lr = 5.0e-5`).
  - `embed_tokens` et `lm_head` restent gelés et liés (`tie_word_embeddings=True`) : fusion `merge_and_unload()` 100 % propre sans dédoublement !
- **Mélange intra-batch garanti** : chaque shard téléchargé contient déjà `35 % bci / 35 % dyu / 18 % fr / 12 % en` mélangés aléatoirement (`0 % nch` vérifié par assertion).
- **Relais automatique résilient** :
  - Reprise exacte au shard et à la ligne (`etat_relais.json` + `etat_optimiseur.pt`),
  - Upload atomique sur Hugging Face après chaque shard terminé (et si watchdog déclenché) + compression `super_squash_history` pour zéro saturation LFS.

## 1. Environnement AMD ROCm (sans écraser PyTorch HIP)

In [ ]:
import os

# Allocateur ROCm anti-fragmentation, logs MIOpen réduits et upload LFS standard (évite le conflit Xet _create_progress_bar)
os.environ["PYTORCH_HIP_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["MIOPEN_LOG_LEVEL"] = "3"
os.environ["MIOPEN_ENABLE_LOGGING"] = "0"
os.environ["MIOPEN_ENABLE_LOGGING_CMD"] = "0"

In [ ]:
%pip install --quiet --upgrade pip
%pip install --quiet --force-reinstall --no-cache-dir "transformers==4.57.3" "tokenizers==0.22.2" "huggingface_hub>=0.34.0,<1.0.0"
%pip install --quiet --upgrade-strategy only-if-needed "numpy>=2.0.0,<2.8.0" "pyarrow>=17.0.0" "peft>=0.14.0" "accelerate>=0.34.0" "safetensors" "tqdm"

In [ ]:
import getpass
import inspect
import json
import math
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import peft
import pyarrow.parquet as pq
import torch
import transformers
from huggingface_hub import CommitOperationAdd, HfApi, hf_hub_download
from huggingface_hub.errors import RepositoryNotFoundError
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

HEURE_DEBUT_SESSION = time.monotonic()
Path("/root/.local/share/jupyter").mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("[ERREUR] Aucun GPU détecté (CUDA / AMD ROCm HIP requis).")

est_rocm = hasattr(torch.version, "hip") and torch.version.hip is not None
bf16_ok = torch.cuda.is_bf16_supported()
if not bf16_ok:
    raise RuntimeError("[ERREUR] Le support bfloat16 natif est requis pour Llama 3.2 1B sur AMD MI3xx.")

nom_gpu = torch.cuda.get_device_name(0)
vram_go = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print("=" * 72)
print(f"[OK] GPU actif         : {nom_gpu} ({vram_go:.1f} Go VRAM)")
print(f"Backend           : {'AMD ROCm HIP ' + str(torch.version.hip) if est_rocm else 'CUDA'}")
print(f"Transformers      : {transformers.__version__} | PEFT : {peft.__version__}")
print("=" * 72)

## 2. Configuration Centralisée du Relais & Hyperparamètres (`Llama 3.2 1B` • `60M`)

In [ ]:
# Authentification Hugging Face
JETON_HF = os.getenv("HF_TOKEN", "").strip()
if not JETON_HF and Path(".env").exists():
    for ligne_env in Path(".env").read_text(encoding="utf-8").splitlines():
        if ligne_env.strip().startswith("HF_TOKEN="):
            JETON_HF = ligne_env.split("=", 1)[1].strip().strip('"').strip("'")
            break
if not JETON_HF:
    JETON_HF = getpass.getpass("Jeton Hugging Face (lecture + écriture Tree-AI-lab + accès Llama 3.2) : ").strip()
if not JETON_HF:
    raise PermissionError("[ERREUR] Jeton HF obligatoire.")
os.environ["HF_TOKEN"] = JETON_HF

# Dépôts Hugging Face
IDENTIFIANT_MODELE_BASE = "meta-llama/Llama-3.2-1B"
DEPOT_SHARDS_HF = "Tree-AI-lab/kouman-llama32-1b-cpt-shards"
DEPOT_CHECKPOINT_HF = "issoufzousko07/kouman-llama32-1b-cpt"
DEPOT_REPRISE_INITIALE_HF = "Tree-AI-lab/kouman-llama32-1b-cpt"
DEPOT_CHECKPOINT_PRIVE = True

# Configuration LoRA + RMSNorm = 59 951 104 (~60M) paramètres entraînables sur Llama 3.2 1B
# Calcul exact : 16 couches * 44 032 * 85 = 59 883 520 (LoRA) + 33 * 2048 = 67 584 (RMSNorm) = 59 951 104
LORA_R = 85
LORA_ALPHA = 170
LORA_DROPOUT = 0.0
MODULES_LORA = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]
MODULES_NORMS_SAUVEGARDE = ["input_layernorm", "post_attention_layernorm", "norm"]

# Dimensionnement adaptatif VRAM (lot effectif strictement constant = 128 séquences = 262 144 tokens / step)
LONGUEUR_SEQUENCE = 2048
if vram_go >= 100.0:
    TAILLE_MICRO_LOT = 32
    PAS_ACCUMULATION = 4
elif vram_go >= 40.0:
    TAILLE_MICRO_LOT = 16
    PAS_ACCUMULATION = 8
else:
    TAILLE_MICRO_LOT = 4
    PAS_ACCUMULATION = 32
NB_WORKERS_DATALOADER = min(4, os.cpu_count() or 2)
ACTIVER_TORCH_COMPILE = False  # Mettre True sur sessions longues pour +35% supplémentaires (warmup ~30s au pas 1)

# Taux d'apprentissage différenciés (rsLoRA + RMSNorm)
TAUX_APPRENTISSAGE_LORA = 2.0e-4
TAUX_APPRENTISSAGE_NORMS = 5.0e-5
DECROISSANCE_POIDS_LORA = 0.01
CLIPPING_NORME_GRADIENT = 1.0

# Scheduler WSD (Warmup-Stable-Decay) compatible relais illimités
# - "warmup_stable" : toutes les sessions intermédiaires
# - "decay"         : phase finale avant fusion (~5 derniers shards = 600 steps, arrêt auto puis fusion Cellule 8)
PHASE_SCHEDULER = "decay"
PAS_WARMUP = 250
PAS_DECAY_TOTAL = 600

# Sauvegarde directe sur HF dès qu'UN SEUL shard est terminé (+ watchdog optionnel)
DUREE_MAX_ENTRAINEMENT_MINUTES = None  # Mettre 46.0 si session limitée à 1h stricte
INTERVALLE_LOG_STEPS = 5
LANGUES_INTERDITES = {"nch"}

DOSSIER_TRAVAIL = Path("travail_cpt_llama32_1b_relais")
DOSSIER_CHECKPOINT_LOCAL = DOSSIER_TRAVAIL / "checkpoint_courant"
DOSSIER_CACHE_SHARDS = DOSSIER_TRAVAIL / "shards_cache"
DOSSIER_TRAVAIL.mkdir(parents=True, exist_ok=True)
DOSSIER_CHECKPOINT_LOCAL.mkdir(parents=True, exist_ok=True)
DOSSIER_CACHE_SHARDS.mkdir(parents=True, exist_ok=True)

print(f"Modèle base      : {IDENTIFIANT_MODELE_BASE}")
print(f"Shards source    : {DEPOT_SHARDS_HF}")
print(f"Relais HF        : {DEPOT_CHECKPOINT_HF}")
print(f"Lot effectif     : {TAILLE_MICRO_LOT} × {PAS_ACCUMULATION} = {TAILLE_MICRO_LOT * PAS_ACCUMULATION} séquences ({TAILLE_MICRO_LOT * PAS_ACCUMULATION * LONGUEUR_SEQUENCE:,} tokens/step)")

## 3. Contrats Anti-Fallback, Manifeste Shards & Reprise Relais HF

In [ ]:
class ErreurValidationShards(RuntimeError):
    """Levée si les shards pré-tokenisés sont absents ou incompatibles."""


class ErreurNouchiDetecte(ValueError):
    """Levée si la langue interdite 'nch' est détectée dans un shard."""


class ErreurRepriseCheckpoint(RuntimeError):
    """Levée si la reprise d'un checkpoint relais existant échoue."""


client_hf = HfApi(token=JETON_HF)


def charger_manifeste_shards() -> dict:
    """Télécharge et valide le manifeste généré par le Notebook 1 (Llama 3.2 1B)."""
    try:
        chemin_local = hf_hub_download(
            repo_id=DEPOT_SHARDS_HF,
            filename="manifeste_shards.json",
            repo_type="dataset",
            token=JETON_HF,
            force_download=True,
        )
    except Exception as err:
        raise ErreurValidationShards(
            f"[ERREUR] Impossible de lire manifeste_shards.json dans {DEPOT_SHARDS_HF}. "
            "Exécutez d'abord le Notebook 1 (kouman_llama32_1b_pretokenisation_shards.ipynb) !"
        ) from err

    manifeste = json.loads(Path(chemin_local).read_text(encoding="utf-8"))
    if manifeste.get("modele_tokenizer") != IDENTIFIANT_MODELE_BASE:
        raise ErreurValidationShards(
            f"[ERREUR] Tokenizer du manifeste incompatible : {manifeste.get('modele_tokenizer')} != {IDENTIFIANT_MODELE_BASE}"
        )
    if manifeste.get("longueur_sequence") != LONGUEUR_SEQUENCE:
        raise ErreurValidationShards(
            f"[ERREUR] Longueur de séquence incompatible : {manifeste.get('longueur_sequence')} != {LONGUEUR_SEQUENCE}"
        )
    shards = manifeste.get("shards_publies", [])
    if not shards:
        raise ErreurValidationShards("[ERREUR] Aucun shard d'entraînement publié dans manifeste_shards.json !")

    print(
        f"Manifeste shards OK : {len(shards)} shard(s) disponible(s) | "
        f"{manifeste.get('tokens_totaux_publies', 0) / 1e9:.3f}B tokens pré-tokenisés | "
        f"termine={manifeste.get('termine')}"
    )
    return manifeste


def _lister_fichiers_depot(depot_hf: str) -> set[str]:
    try:
        return set(client_hf.list_repo_files(repo_id=depot_hf, repo_type="model", token=JETON_HF))
    except RepositoryNotFoundError as err:
        raise ErreurRepriseCheckpoint(f"[ERREUR] Dépôt checkpoint introuvable : {depot_hf}") from err


def inspecter_etat_relais_hf() -> dict | None:
    """Retourne l'état relais s'il existe sur HF, None s'il s'agit d'un tout premier run."""
    client_hf.create_repo(
        repo_id=DEPOT_CHECKPOINT_HF,
        repo_type="model",
        private=DEPOT_CHECKPOINT_PRIVE,
        exist_ok=True,
    )
    depot_source = DEPOT_CHECKPOINT_HF
    fichiers_repo = _lister_fichiers_depot(depot_source)

    if (
        "etat_relais.json" not in fichiers_repo
        and DEPOT_REPRISE_INITIALE_HF
        and DEPOT_REPRISE_INITIALE_HF != DEPOT_CHECKPOINT_HF
        and client_hf.repo_exists(repo_id=DEPOT_REPRISE_INITIALE_HF, repo_type="model", token=JETON_HF)
    ):
        fichiers_historique = _lister_fichiers_depot(DEPOT_REPRISE_INITIALE_HF)
        if "etat_relais.json" in fichiers_historique:
            print(
                f"Amorçage depuis {DEPOT_REPRISE_INITIALE_HF} "
                f"(les nouveaux relais seront poussés sur {DEPOT_CHECKPOINT_HF})."
            )
            depot_source = DEPOT_REPRISE_INITIALE_HF
            fichiers_repo = fichiers_historique

    if "etat_relais.json" not in fichiers_repo:
        print("Aucun relais précédent trouvé : initialisation d'un entraînement neuf (step 0).")
        return None

    requis = {
        "checkpoint_courant/adapter_config.json",
        "checkpoint_courant/adapter_model.safetensors",
        "checkpoint_courant/etat_optimiseur.pt",
    }
    manquants = [f for f in sorted(requis) if f not in fichiers_repo]
    if manquants:
        raise ErreurRepriseCheckpoint(
            f"[ERREUR] Relais corrompu sur {depot_source} : etat_relais.json présent mais fichiers manquants : {manquants}"
        )

    chemin_etat = hf_hub_download(
        repo_id=depot_source,
        filename="etat_relais.json",
        repo_type="model",
        token=JETON_HF,
        force_download=True,
    )
    etat = json.loads(Path(chemin_etat).read_text(encoding="utf-8"))

    if "historique_metriques.jsonl" in fichiers_repo:
        chemin_hist_hf = hf_hub_download(
            repo_id=depot_source,
            filename="historique_metriques.jsonl",
            repo_type="model",
            token=JETON_HF,
            force_download=True,
        )
        (DOSSIER_TRAVAIL / "historique_metriques.jsonl").write_bytes(Path(chemin_hist_hf).read_bytes())

    for chemin_repo in sorted(fichiers_repo):
        if chemin_repo.startswith("checkpoint_courant/"):
            relatif = chemin_repo.removeprefix("checkpoint_courant/")
            cible = DOSSIER_CHECKPOINT_LOCAL / relatif
            cible.parent.mkdir(parents=True, exist_ok=True)
            telecharge = hf_hub_download(
                repo_id=depot_source,
                filename=chemin_repo,
                repo_type="model",
                token=JETON_HF,
                force_download=True,
            )
            cible.write_bytes(Path(telecharge).read_bytes())

    print(
        f"Relais repris depuis {depot_source} : step_global={etat['step_global']} | "
        f"shard={etat['indice_shard']} | ligne={etat['ligne_dans_shard']} | "
        f"tokens_vus={etat['tokens_entraines'] / 1e9:.3f}B"
    )
    return etat


MANIFESTE_SHARDS = charger_manifeste_shards()
etat_relais_precedent = inspecter_etat_relais_hf()

## 4. Chargement Llama 3.2 1B + PEFT LoRA `r=85` & RMSNorm (`~60M` Paramètres Entraînables)

- Attention **`sdpa`** (PyTorch FlashAttention natif AMD ROCm) : zéro overhead, vitesse maximale sur MI300X / MI325X.
- Assertion stricte **`55M <= N <= 65M`** (`59 951 104` attendus exactement) pour garantir le budget exact de **60M paramètres entraînables**.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(IDENTIFIANT_MODELE_BASE, token=JETON_HF)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Chargement du modèle base {IDENTIFIANT_MODELE_BASE} en bfloat16 (attn=sdpa)...")
kwargs_modele = {
    "attn_implementation": "sdpa",
    "token": JETON_HF,
}
if "dtype" in inspect.signature(AutoModelForCausalLM.from_pretrained).parameters:
    kwargs_modele["dtype"] = torch.bfloat16
else:
    kwargs_modele["torch_dtype"] = torch.bfloat16

modele_base = AutoModelForCausalLM.from_pretrained(IDENTIFIANT_MODELE_BASE, **kwargs_modele)
modele_base.config.use_cache = False

if vram_go < 40.0:
    print(f"GPU {vram_go:.0f} Go détecté (< 40 Go) : activation du Gradient Checkpointing.")
    modele_base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    modele_base.enable_input_require_grads()

if etat_relais_precedent is None:
    config_lora = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        use_rslora=True,
        bias="none",
        target_modules=MODULES_LORA,
        modules_to_save=MODULES_NORMS_SAUVEGARDE,
        task_type=TaskType.CAUSAL_LM,
    )
    modele = get_peft_model(modele_base, config_lora)
else:
    modele = PeftModel.from_pretrained(
        modele_base,
        str(DOSSIER_CHECKPOINT_LOCAL),
        is_trainable=True,
    )

modele = modele.to(device="cuda", dtype=torch.bfloat16)
if ACTIVER_TORCH_COMPILE:
    print("Compilation torch.compile du modèle Llama 3.2 1B...")
    modele = torch.compile(modele)

params_totaux = sum(p.numel() for p in modele.parameters())
params_entrainables = sum(p.numel() for p in modele.parameters() if p.requires_grad)
ratio_pct = 100.0 * params_entrainables / params_totaux

print(f"Paramètres totaux       : {params_totaux:,}")
print(f"Paramètres entraînables : {params_entrainables:,} ({params_entrainables / 1e6:.2f} M | {ratio_pct:.2f} %)")

if not (55_000_000 <= params_entrainables <= 65_000_000):
    raise AssertionError(
        f"[ERREUR] Nombre de paramètres entraînables hors cible 60M ({params_entrainables:,}). "
        "Attendu : ~59,95M (LoRA r=85 + RMSNorm sur Llama-3.2-1B)."
    )

if etat_relais_precedent is not None:
    attendu = etat_relais_precedent.get("params_entrainables")
    if attendu is not None and int(attendu) != int(params_entrainables):
        raise ErreurRepriseCheckpoint(
            f"[ERREUR] Nombre de paramètres différent du checkpoint précédent : {params_entrainables} != {attendu}"
        )

## 5. Optimiseur Différencié (`LoRA` + `RMSNorm`) & Scheduler WSD (`Warmup-Stable-Decay`)

In [ ]:
params_norms = []
params_lora = []
noms_parametres_vus = set()

for nom, param in modele.named_parameters():
    if not param.requires_grad:
        continue
    if id(param) in noms_parametres_vus:
        continue
    noms_parametres_vus.add(id(param))
    if "norm" in nom.lower():
        params_norms.append(param)
    else:
        params_lora.append(param)

if not params_lora:
    raise RuntimeError("[ERREUR] Aucun paramètre LoRA entraînable trouvé.")

groupes_optimiseur = [
    {
        "params": params_lora,
        "lr": TAUX_APPRENTISSAGE_LORA,
        "weight_decay": DECROISSANCE_POIDS_LORA,
        "lr_cible_base": TAUX_APPRENTISSAGE_LORA,
    },
]
if params_norms:
    groupes_optimiseur.append(
        {
            "params": params_norms,
            "lr": TAUX_APPRENTISSAGE_NORMS,
            "weight_decay": 0.0,
            "lr_cible_base": TAUX_APPRENTISSAGE_NORMS,
        }
    )

try:
    optimiseur = torch.optim.AdamW(groupes_optimiseur, betas=(0.9, 0.95), eps=1e-8, fused=True)
    print("Optimiseur AdamW fused=True actif sur GPU.")
except Exception:
    optimiseur = torch.optim.AdamW(groupes_optimiseur, betas=(0.9, 0.95), eps=1e-8)

if etat_relais_precedent is not None:
    chemin_opt = DOSSIER_CHECKPOINT_LOCAL / "etat_optimiseur.pt"
    etat_opt = torch.load(chemin_opt, map_location="cpu", weights_only=True)
    optimiseur.load_state_dict(etat_opt["optimizer_state_dict"])
    step_global = int(etat_relais_precedent["step_global"])
    indice_shard_courant = int(etat_relais_precedent["indice_shard"])
    ligne_dans_shard_courant = int(etat_relais_precedent["ligne_dans_shard"])
    tokens_entraines = int(etat_relais_precedent["tokens_entraines"])
    step_debut_decay = etat_relais_precedent.get("step_debut_decay")
    print(f"États AdamW restaurés au step {step_global:,} !")
else:
    step_global = 0
    indice_shard_courant = 0
    ligne_dans_shard_courant = 0
    tokens_entraines = 0
    step_debut_decay = None

if PHASE_SCHEDULER == "decay" and step_debut_decay is None:
    step_debut_decay = step_global
    print(f"Activation de la phase DECAY WSD à partir du step {step_debut_decay}.")


def appliquer_facteur_wsd(step_actuel: int) -> float:
    """Calcule le multiplicateur LR Warmup-Stable-Decay et l'applique à l'optimiseur."""
    global step_debut_decay
    if step_actuel < PAS_WARMUP:
        facteur = float(step_actuel + 1) / float(max(1, PAS_WARMUP))
    elif PHASE_SCHEDULER == "warmup_stable":
        facteur = 1.0
    elif PHASE_SCHEDULER == "decay":
        if step_debut_decay is None:
            step_debut_decay = step_actuel
            print(
                f"Activation automatique de la phase DECAY WSD au step {step_debut_decay:,} "
                f"(fin prévue au step {step_debut_decay + PAS_DECAY_TOTAL:,})."
            )
        avancement = min(1.0, max(0.0, (step_actuel - step_debut_decay) / float(max(1, PAS_DECAY_TOTAL))))
        facteur = 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * avancement))
    else:
        raise ValueError(f"[ERREUR] PHASE_SCHEDULER inconnue : {PHASE_SCHEDULER}")

    for groupe in optimiseur.param_groups:
        groupe["lr"] = groupe["lr_cible_base"] * facteur
    return facteur

## 6. Chargement des Shards & Évaluation Multilingue Figée

In [ ]:
class JeuShardPret(Dataset):
    """Dataset PyTorch en mémoire zéro-copie pour un micro-shard Parquet (int32 -> int64)."""

    def __init__(self, chemin_parquet: Path, ligne_depart: int = 0):
        table = pq.read_table(chemin_parquet, columns=["input_ids", "lang"])
        langues = table["lang"].to_pylist()
        if "nch" in set(langues):
            raise ErreurNouchiDetecte(f"[ERREUR] Shard contaminé par du Nouchi ('nch') : {chemin_parquet}")

        tenseur_complet = torch.tensor(table["input_ids"].to_pylist(), dtype=torch.int64)
        if tenseur_complet.ndim != 2 or tenseur_complet.shape[1] != LONGUEUR_SEQUENCE:
            raise ErreurValidationShards(
                f"[ERREUR] Dimensions shard invalides : {tuple(tenseur_complet.shape)} != (*, {LONGUEUR_SEQUENCE})"
            )
        if ligne_depart > len(tenseur_complet):
            raise ErreurRepriseCheckpoint(
                f"[ERREUR] ligne_depart ({ligne_depart}) > taille du shard ({len(tenseur_complet)})"
            )
        self.sequences = tenseur_complet[ligne_depart:]

    def __len__(self) -> int:
        return self.sequences.shape[0]

    def __getitem__(self, idx: int) -> torch.Tensor:
        return self.sequences[idx]


def obtenir_shard_local(chemin_repo_shard: str) -> Path:
    telecharge = hf_hub_download(
        repo_id=DEPOT_SHARDS_HF,
        filename=chemin_repo_shard,
        repo_type="dataset",
        token=JETON_HF,
        local_dir=str(DOSSIER_CACHE_SHARDS),
    )
    return Path(telecharge)


def preparer_batches_eval() -> dict[str, torch.Tensor]:
    chemin_eval = obtenir_shard_local("eval/eval_multilingue.parquet")
    table_eval = pq.read_table(chemin_eval, columns=["input_ids", "lang"])
    toutes_seqs = table_eval["input_ids"].to_pylist()
    toutes_langs = table_eval["lang"].to_pylist()
    if "nch" in set(toutes_langs):
        raise ErreurNouchiDetecte("[ERREUR] Nouchi détecté dans eval_multilingue.parquet !")

    par_langue: dict[str, list[list[int]]] = {}
    for seq, lg in zip(toutes_seqs, toutes_langs):
        par_langue.setdefault(lg, []).append(seq)

    return {lg: torch.tensor(seqs, dtype=torch.int64) for lg, seqs in par_langue.items()}


EVAL_PAR_LANGUE = preparer_batches_eval()
print("Jeu d'évaluation figé chargé :", {lg: t.shape[0] for lg, t in EVAL_PAR_LANGUE.items()})


@torch.no_grad()
def evaluer_perplexite_par_langue(max_seqs_par_langue: int = 128) -> dict[str, dict[str, float]]:
    modele.eval()
    resultats = {}
    batch_eval = min(8, TAILLE_MICRO_LOT)
    for lg, tenseur in EVAL_PAR_LANGUE.items():
        sous_ensemble = tenseur[:max_seqs_par_langue]
        pertes = []
        for debut in range(0, len(sous_ensemble), batch_eval):
            lot = sous_ensemble[debut:debut + batch_eval].to("cuda", non_blocking=True)
            with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                sortie = modele(input_ids=lot, labels=lot)
            pertes.append(float(sortie.loss.detach().cpu()))
            del lot, sortie
        loss_moy = sum(pertes) / len(pertes)
        resultats[lg] = {
            "loss": round(loss_moy, 4),
            "ppl": round(math.exp(min(loss_moy, 20.0)), 2),
        }
    torch.cuda.empty_cache()
    modele.train()
    return resultats

## 7. Fonction de Sauvegarde Atomique Relais & Boucle d'Entraînement

In [ ]:
def temps_ecoule_minutes() -> float:
    return (time.monotonic() - HEURE_DEBUT_SESSION) / 60.0


class ErreurDivergenceModele(RuntimeError):
    """Levée si la perte ou la perplexité diverge — bloque tout écrasement sur HF."""


def verifier_sante_metriques_eval(metriques_eval: dict, contexte: str) -> None:
    """Refuse tout état divergé ou NaN/Inf après warmup afin de ne jamais écraser HF."""
    if step_global < 500:
        return
    for lg, vals in metriques_eval.items():
        perte = vals.get("loss")
        ppl = vals.get("ppl")
        if perte is None or ppl is None or not math.isfinite(perte) or not math.isfinite(ppl) or ppl > 10.0:
            raise ErreurDivergenceModele(
                f"[ARRET] DIVERGENCE DÉTECTÉE ({contexte}) à step={step_global:,} : "
                f"{lg} -> loss={perte}, ppl={ppl} (seuil max PPL=10.0). "
                "Upload HF BLOQUÉ pour protéger le dernier bon checkpoint !"
            )


def sauvegarder_et_pousser_relais(
    indice_shard: int,
    ligne_dans_shard: int,
    motif_arret: str,
    metriques_eval: dict,
) -> None:
    """Sauvegarde adaptateurs + RMSNorm + AdamW + curseur et pousse atomiquement sur HF."""
    verifier_sante_metriques_eval(metriques_eval, contexte=f"avant_push_hf:{motif_arret}")
    print(f"Sauvegarde relais ({motif_arret}) à step={step_global:,} | shard={indice_shard} | ligne={ligne_dans_shard}...")

    modele.save_pretrained(str(DOSSIER_CHECKPOINT_LOCAL))
    tokenizer.save_pretrained(str(DOSSIER_CHECKPOINT_LOCAL))

    torch.save(
        {"optimizer_state_dict": optimiseur.state_dict()},
        DOSSIER_CHECKPOINT_LOCAL / "etat_optimiseur.pt",
    )

    etat_relais = {
        "modele_base": IDENTIFIANT_MODELE_BASE,
        "step_global": step_global,
        "indice_shard": indice_shard,
        "ligne_dans_shard": ligne_dans_shard,
        "tokens_entraines": tokens_entraines,
        "params_entrainables": params_entrainables,
        "lora_r": LORA_R,
        "phase_scheduler": PHASE_SCHEDULER,
        "step_debut_decay": step_debut_decay,
        "motif_dernier_arret": motif_arret,
        "dernieres_metriques_eval": metriques_eval,
        "horodatage_utc": datetime.now(timezone.utc).isoformat(),
    }
    chemin_etat_local = DOSSIER_TRAVAIL / "etat_relais.json"
    chemin_etat_local.write_text(json.dumps(etat_relais, indent=2, ensure_ascii=False), encoding="utf-8")

    chemin_hist = DOSSIER_TRAVAIL / "historique_metriques.jsonl"
    with open(chemin_hist, "a", encoding="utf-8") as f_hist:
        f_hist.write(
            json.dumps(
                {
                    "step_global": step_global,
                    "tokens_entraines": tokens_entraines,
                    "motif": motif_arret,
                    "eval": metriques_eval,
                    "utc": etat_relais["horodatage_utc"],
                },
                ensure_ascii=False,
            )
            + "\n"
        )

    operations = []
    for fichier_ckpt in sorted(DOSSIER_CHECKPOINT_LOCAL.rglob("*")):
        if fichier_ckpt.is_file():
            relatif = fichier_ckpt.relative_to(DOSSIER_CHECKPOINT_LOCAL).as_posix()
            operations.append(
                CommitOperationAdd(
                    path_or_fileobj=str(fichier_ckpt),
                    path_in_repo=f"checkpoint_courant/{relatif}",
                )
            )
    operations.append(CommitOperationAdd(path_or_fileobj=str(chemin_etat_local), path_in_repo="etat_relais.json"))
    operations.append(CommitOperationAdd(path_or_fileobj=str(chemin_hist), path_in_repo="historique_metriques.jsonl"))

    commits_existants = client_hf.list_repo_commits(repo_id=DEPOT_CHECKPOINT_HF, repo_type="model")
    if len(commits_existants) > 1:
        client_hf.super_squash_history(
            repo_id=DEPOT_CHECKPOINT_HF,
            repo_type="model",
            branch="main",
            commit_message="squash historique avant nouveau relais",
        )

    message_commit = (
        f"checkpoint atomique: step {step_global} "
        f"({tokens_entraines / 1e9:.3f}B tokens) [{motif_arret}]"
    )
    # Force l'upload LFS standard (désactive Xet même si un reliquat huggingface_hub 1.x est présent)
    os.environ["HF_HUB_DISABLE_XET"] = "1"
    try:
        import huggingface_hub.constants as _hf_const
        import huggingface_hub._commit_api as _hf_commit
        _hf_const.HF_HUB_DISABLE_XET = True
        if hasattr(_hf_commit, "is_xet_available"):
            _hf_commit.is_xet_available = lambda: False
    except Exception:
        pass

    client_hf.create_commit(
        repo_id=DEPOT_CHECKPOINT_HF,
        repo_type="model",
        operations=operations,
        commit_message=message_commit,
    )
    client_hf.super_squash_history(
        repo_id=DEPOT_CHECKPOINT_HF,
        repo_type="model",
        branch="main",
        commit_message=message_commit,
    )
    print(f"Historique LFS compressé sur HF (1 commit conservé pour step={step_global}).")
    print(f"Relais synchronisé atomiquement sur HF ({temps_ecoule_minutes():.1f} min écoulées).")


if step_global == 0:
    print("Mesure baseline avant le premier step d'entraînement...")
    eval_initiale = evaluer_perplexite_par_langue()
    print("Perplexité initiale Llama 3.2 1B :", eval_initiale)
else:
    print(f"Contrôle santé rapide du checkpoint repris (step={step_global:,})...")
    eval_sante_reprise = evaluer_perplexite_par_langue(max_seqs_par_langue=16)
    print("Santé checkpoint repris :", eval_sante_reprise)
    verifier_sante_metriques_eval(eval_sante_reprise, contexte="reprise_checkpoint_hf")

shards_disponibles = MANIFESTE_SHARDS["shards_publies"]
arret_watchdog = False
arret_fin_decay = False
besoin_push_relais = False
modele.train()
optimiseur.zero_grad(set_to_none=True)
compteur_accum = 0
perte_accum_gpu = torch.zeros((), device="cuda", dtype=torch.float32)

if PHASE_SCHEDULER == "decay":
    if step_debut_decay is None:
        step_debut_decay = step_global
        print(
            f"Activation de la phase DECAY WSD au step {step_debut_decay:,} "
            f"(fin prévue au step {step_debut_decay + PAS_DECAY_TOTAL:,} | {PAS_DECAY_TOTAL} steps)."
        )
    else:
        steps_decay_faits = max(0, step_global - step_debut_decay)
        steps_decay_restants = max(0, PAS_DECAY_TOTAL - steps_decay_faits)
        print(
            f"Reprise de la phase DECAY WSD : {steps_decay_faits}/{PAS_DECAY_TOTAL} steps faits "
            f"(début={step_debut_decay:,} | actuel={step_global:,} | restant={steps_decay_restants} steps)."
        )
else:
    print(f"Phase scheduler active : {PHASE_SCHEDULER} (step={step_global:,} | LR={optimiseur.param_groups[0]['lr']:.2e}).")

def perte_ce_par_blocs(modele_actif, lot_ids: torch.Tensor, taille_bloc_tokens: int = 4096) -> torch.Tensor:
    """
    Calcule lm_head + cross_entropy par blocs de 4096 tokens avec gradient checkpointing :
    évite l'allocation de 32+ Go pour [B, 2048, 128256] en FP32 (mathématiquement 100% identique).
    """
    base_causale = modele_actif.get_base_model() if hasattr(modele_actif, "get_base_model") else modele_actif
    sortie_corps = base_causale.model(input_ids=lot_ids, use_cache=False)
    etats = sortie_corps.last_hidden_state[:, :-1, :].reshape(-1, base_causale.config.hidden_size)
    cibles = lot_ids[:, 1:].reshape(-1)
    nb_tokens = cibles.numel()

    def _ce_bloc(h_bloc: torch.Tensor, y_bloc: torch.Tensor) -> torch.Tensor:
        logits_bloc = base_causale.lm_head(h_bloc)
        return torch.nn.functional.cross_entropy(logits_bloc.float(), y_bloc, reduction="sum")

    perte_totale = torch.zeros((), device=lot_ids.device, dtype=torch.float32)
    for debut in range(0, nb_tokens, taille_bloc_tokens):
        fin = min(debut + taille_bloc_tokens, nb_tokens)
        perte_bloc = torch.utils.checkpoint.checkpoint(
            _ce_bloc,
            etats[debut:fin],
            cibles[debut:fin],
            use_reentrant=False,
        )
        perte_totale = perte_totale + perte_bloc
    return perte_totale / nb_tokens


while indice_shard_courant < len(shards_disponibles):
    if DUREE_MAX_ENTRAINEMENT_MINUTES is not None and temps_ecoule_minutes() >= DUREE_MAX_ENTRAINEMENT_MINUTES:
        arret_watchdog = True
        break

    info_shard = shards_disponibles[indice_shard_courant]
    chemin_parquet_shard = obtenir_shard_local(info_shard["fichier"])
    nb_total_lignes_shard = int(info_shard["nb_sequences"])

    jeu_shard = JeuShardPret(chemin_parquet_shard, ligne_depart=ligne_dans_shard_courant)
    chargeur = DataLoader(
        jeu_shard,
        batch_size=TAILLE_MICRO_LOT,
        shuffle=False,
        num_workers=NB_WORKERS_DATALOADER,
        pin_memory=True,
        persistent_workers=(NB_WORKERS_DATALOADER > 0),
        prefetch_factor=(4 if NB_WORKERS_DATALOADER > 0 else None),
        drop_last=False,
    )

    print(
        f"Shard {indice_shard_courant + 1}/{len(shards_disponibles)} : {info_shard['fichier']} "
        f"(à partir de la ligne {ligne_dans_shard_courant}/{nb_total_lignes_shard})"
    )
    barre = tqdm(chargeur, desc=f"Shard {indice_shard_courant}", leave=False)

    for lot_ids in barre:
        nb_seqs_lot = int(lot_ids.shape[0])
        lot_cuda = lot_ids.to("cuda", non_blocking=True)

        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            perte_brute = perte_ce_par_blocs(modele, lot_cuda, taille_bloc_tokens=4096)
            perte_normalisee = perte_brute / PAS_ACCUMULATION

        perte_normalisee.backward()
        # Accumulation sur GPU (zéro synchronisation CPU/GPU sur les micro-lots !)
        perte_accum_gpu.add_(perte_brute.detach().float())
        compteur_accum += 1
        ligne_dans_shard_courant += nb_seqs_lot
        tokens_entraines += nb_seqs_lot * LONGUEUR_SEQUENCE

        if compteur_accum >= PAS_ACCUMULATION:
            facteur_lr = appliquer_facteur_wsd(step_global)
            torch.nn.utils.clip_grad_norm_(
                [p for g in optimiseur.param_groups for p in g["params"]],
                CLIPPING_NORME_GRADIENT,
            )
            optimiseur.step()
            optimiseur.zero_grad(set_to_none=True)

            step_global += 1
            if step_global % INTERVALLE_LOG_STEPS == 0:
                loss_step = (perte_accum_gpu / compteur_accum).item()
                lr_actuel = optimiseur.param_groups[0]["lr"]
                barre.set_postfix(
                    step=step_global,
                    loss=f"{loss_step:.3f}",
                    lr=f"{lr_actuel:.2e}",
                    min=f"{temps_ecoule_minutes():.1f}",
                )
                if step_global >= 500 and (not math.isfinite(loss_step) or loss_step > 4.5):
                    raise ErreurDivergenceModele(
                        f"[ARRET] DIVERGENCE EN COURS D'ENTRAÎNEMENT à step={step_global:,} "
                        f"(shard={indice_shard_courant}, ligne={ligne_dans_shard_courant}) : "
                        f"loss_step={loss_step:.4f} (> 4.5) | lr={lr_actuel:.2e}. "
                        "Arrêt d'urgence sans rien pousser sur HF !"
                    )
            perte_accum_gpu.zero_()
            compteur_accum = 0

            if (
                PHASE_SCHEDULER == "decay"
                and step_debut_decay is not None
                and (step_global - step_debut_decay) >= PAS_DECAY_TOTAL
            ):
                print(
                    f"Objectif DECAY WSD atteint ({step_global - step_debut_decay}/{PAS_DECAY_TOTAL} steps, "
                    f"LR={optimiseur.param_groups[0]['lr']:.2e}). Sauvegarde finale..."
                )
                arret_fin_decay = True
                break

            if DUREE_MAX_ENTRAINEMENT_MINUTES is not None and temps_ecoule_minutes() >= DUREE_MAX_ENTRAINEMENT_MINUTES:
                print(
                    f"Watchdog atteint à {temps_ecoule_minutes():.1f} min. Sauvegarde relais..."
                )
                arret_watchdog = True
                break

    if arret_fin_decay:
        optimiseur.zero_grad(set_to_none=True)
        compteur_accum = 0
        perte_accum_gpu.zero_()
        eval_finale_decay = evaluer_perplexite_par_langue()
        print("Évaluation finale post-DECAY :", eval_finale_decay)
        sauvegarder_et_pousser_relais(
            indice_shard=indice_shard_courant,
            ligne_dans_shard=ligne_dans_shard_courant,
            motif_arret="fin_decay_cpt",
            metriques_eval=eval_finale_decay,
        )
        besoin_push_relais = False
        print(
            f"Phase DECAY WSD terminée ({step_global - step_debut_decay} steps >= {PAS_DECAY_TOTAL}) ! "
            "Le CPT Llama 3.2 1B est prêt pour la fusion (section 8)."
        )
        break

    if arret_watchdog:
        optimiseur.zero_grad(set_to_none=True)
        compteur_accum = 0
        perte_accum_gpu.zero_()
        eval_courante = evaluer_perplexite_par_langue()
        print("Évaluation relais :", eval_courante)
        sauvegarder_et_pousser_relais(
            indice_shard=indice_shard_courant,
            ligne_dans_shard=ligne_dans_shard_courant,
            motif_arret="watchdog_1h",
            metriques_eval=eval_courante,
        )
        besoin_push_relais = False
        break

    optimiseur.zero_grad(set_to_none=True)
    compteur_accum = 0
    perte_accum_gpu.zero_()

    indice_shard_courant += 1
    ligne_dans_shard_courant = 0
    chemin_parquet_shard.unlink(missing_ok=True)

    eval_shard = evaluer_perplexite_par_langue()
    info_decay_log = ""
    if PHASE_SCHEDULER == "decay" and step_debut_decay is not None:
        info_decay_log = (
            f" | Decay {step_global - step_debut_decay}/{PAS_DECAY_TOTAL} steps "
            f"(LR={optimiseur.param_groups[0]['lr']:.2e})"
        )
    print(f"[OK] Shard {indice_shard_courant - 1} terminé{info_decay_log} | Éval={eval_shard} -> Upload direct sur HF...")
    est_dernier_shard = indice_shard_courant >= len(shards_disponibles)
    decay_termine = (
        PHASE_SCHEDULER == "decay"
        and step_debut_decay is not None
        and (step_global - step_debut_decay) >= PAS_DECAY_TOTAL
    )
    if decay_termine:
        motif_shard = "fin_decay_cpt"
    elif est_dernier_shard:
        motif_shard = "fin_shards"
    else:
        motif_shard = f"shard_{indice_shard_courant - 1}_termine"

    sauvegarder_et_pousser_relais(
        indice_shard=indice_shard_courant,
        ligne_dans_shard=0,
        motif_arret=motif_shard,
        metriques_eval=eval_shard,
    )
    besoin_push_relais = False

    if decay_termine:
        print(
            f"Phase DECAY WSD terminée ({step_global - step_debut_decay} steps >= {PAS_DECAY_TOTAL}) ! "
            "Le CPT Llama 3.2 1B est prêt pour la fusion (section 8)."
        )
        break

if besoin_push_relais:
    eval_finale_session = evaluer_perplexite_par_langue()
    sauvegarder_et_pousser_relais(
        indice_shard=indice_shard_courant,
        ligne_dans_shard=ligne_dans_shard_courant,
        motif_arret="watchdog_fin_boucle",
        metriques_eval=eval_finale_session,
    )

if not arret_watchdog and indice_shard_courant >= len(shards_disponibles):
    print("Tous les shards disponibles dans manifeste_shards.json ont été entraînés !")

## 7B. Test Qualitatif Comparatif (` Base Llama 3.2 1B` vs ` CPT Kouman Llama 3.2 1B`)

Compare directement le modèle **avant CPT** (adaptateurs LoRA + RMSNorm désactivés en mémoire, **0 Mo VRAM supplémentaire**) et **après CPT** sur des complétions pures (Baoulé, Dioula, Français, Anglais) et du *few-shot* de traduction — sans chat template (modèle de base CPT).

In [ ]:
# Cellule autonome : fonctionne directement après l'entraînement OU dans un nouveau kernel (rechargement depuis HF)
import torch

if "modele" not in globals() or "tokenizer" not in globals():
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer

    depot_base_test = globals().get("IDENTIFIANT_MODELE_BASE", "meta-llama/Llama-3.2-1B")
    depot_ckpt_test = globals().get("DEPOT_CHECKPOINT_HF", "issoufzousko07/kouman-llama32-1b-cpt")
    jeton_test = globals().get("JETON_HF", os.environ.get("HF_TOKEN"))
    print(f"Chargement autonome depuis HF : {depot_base_test} + {depot_ckpt_test}/checkpoint_courant...")
    tokenizer = AutoTokenizer.from_pretrained(depot_base_test, token=jeton_test)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id
    base_tmp = AutoModelForCausalLM.from_pretrained(
        depot_base_test,
        torch_dtype=torch.bfloat16,
        attn_implementation="sdpa",
        token=jeton_test,
    ).to("cuda")
    modele = PeftModel.from_pretrained(
        base_tmp, depot_ckpt_test, subfolder="checkpoint_courant", token=jeton_test, is_trainable=False
    ).to("cuda")

PROMPTS_TEST_CPT = [
    ("Baoulé — Récit village", "Kɛ nzue’n tɔ kpa’n, fie’m be yo klanman, yɛ"),
    ("Baoulé — Conseil école", "Sɛ a kunndɛ kɛ a yo kpa suklu lɔ’n, ɔ fata kɛ"),
    ("Dioula — Vie quotidienne", "Sini sɔgɔma, an bɛna taga baara kɛ foro la ɲɔgɔn fɛ,"),
    ("Dioula — Côte d'Ivoire", "Kɔdiwari ye Afiriki tilebinfan jamana dɔ ye min"),
    ("Français — Non-oubli", "La capitale économique de la Côte d'Ivoire est Abidjan, une ville connue pour"),
    ("Anglais — Non-oubli", "Artificial intelligence is transforming education in West Africa by"),
    (
        "Few-shot FR → DYU",
        "Français : Le riz chaud est prêt.\nDioula : Malo kalaman labɛnna.\n"
        "Français : Les enfants vont à l'école.\nDioula : Denmisɛnw bɛ taga kalanso la.\n"
        "Français : Ma mère prépare une bonne sauce au village.\nDioula :",
    ),
    (
        "Few-shot FR → BCI",
        "Français : Le riz chaud est prêt.\nBaoulé : Avie wɛɛn’n w’a yo kpa.\n"
        "Français : Les enfants vont à l'école.\nBaoulé : Ba’m be kɔ suklu lɔ.\n"
        "Français : Demain matin nous irons travailler au champ.\nBaoulé :",
    ),
]


def generer_completion_cpt(prompt: str, max_tokens: int = 80, temperature: float = 0.35) -> str:
    entrees = tokenizer(prompt, return_tensors="pt").to("cuda")
    nb_in = entrees["input_ids"].shape[1]
    torch.manual_seed(42)
    with torch.no_grad():
        sortie = modele.generate(
            **entrees,
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.9,
            repetition_penalty=1.12,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    ids_gen = sortie[0, nb_in:]
    texte = tokenizer.decode(ids_gen, skip_special_tokens=True)
    # Pour le few-shot traduction, coupe proprement à la fin de la 1re ligne
    if prompt.endswith(("Dioula :", "Baoulé :")):
        texte = texte.split("\n")[0]
    # Coupe proprement au dernier mot complet si max_tokens a tranché un caractère UTF-8 multi-octets (ɛ, ɔ, ɲ)
    if len(ids_gen) >= max_tokens and (texte.endswith("\ufffd") or (texte and texte[-1] not in ".!?\n")):
        texte = texte.rstrip("\ufffd")
        if " " in texte:
            texte = texte.rsplit(" ", 1)[0].rstrip(" ,;:") + "…"
    return texte.strip()


mode_train_avant = modele.training
cache_avant = getattr(modele.config, "use_cache", False)
modele.eval()
modele.config.use_cache = True
if hasattr(modele, "get_base_model"):
    modele.get_base_model().config.use_cache = True

try:
    step_aff = globals().get("step_global", "?")
    shard_aff = globals().get("indice_shard_courant", "?")
    print("═" * 82)
    print(f"TEST COMPARATIF CPT LLAMA 3.2 1B (step={step_aff} | shard={shard_aff})")
    print("═" * 82)

    # 1. Comparaison Perplexité Base vs CPT (si evaluer_perplexite_par_langue est chargé)
    if "evaluer_perplexite_par_langue" in globals():
        with modele.disable_adapter():
            eval_base = evaluer_perplexite_par_langue()
        eval_cpt = evaluer_perplexite_par_langue()
        print("\nPERPLEXITÉ COMPARÉE (plus bas = meilleur) :")
        print(f"{'Langue':10s} | {'Base Llama 3.2 1B':>18s} | {'CPT Kouman 1B':>18s} | {'Gain PPL':>12s}")
        print("-" * 68)
        for lg in ("bci", "dyu", "fr", "en"):
            if lg in eval_base and lg in eval_cpt:
                p0, p1 = eval_base[lg]["ppl"], eval_cpt[lg]["ppl"]
                print(f"{lg.upper():10s} | {p0:18.2f} | {p1:18.2f} | {p1 - p0:+12.2f}")

    # 2. Comparaison des complétions Base vs CPT
    print("\nCOMPLÉTIONS COMPARÉES :")
    for titre, prompt in PROMPTS_TEST_CPT:
        with modele.disable_adapter():
            rep_base = generer_completion_cpt(prompt)
        rep_cpt = generer_completion_cpt(prompt)
        print("\n" + "─" * 82)
        print(f"{titre}")
        print(f"Prompt : {prompt.splitlines()[-1]}")
        print(f"Base   : {rep_base}")
        print(f"CPT    : {rep_cpt}")
finally:
    modele.config.use_cache = cache_avant
    if hasattr(modele, "get_base_model"):
        modele.get_base_model().config.use_cache = cache_avant
    if mode_train_avant:
        modele.train()

## 8. Fusion Finale Automatique LoRA + RMSNorm & Export `bfloat16` (déclenchée uniquement après fin du `decay`)

In [ ]:
FUSIONNER_ET_PUBLIER_MODELE_FINAL = True  # S'exécute automatiquement dès que la phase DECAY est terminée
DEPOT_MODELE_FUSIONNE = "issoufzousko07/kouman-llama32-1b-cpt-merged"

decay_est_fini = (
    PHASE_SCHEDULER == "decay"
    and step_debut_decay is not None
    and (step_global - step_debut_decay) >= PAS_DECAY_TOTAL
)

if FUSIONNER_ET_PUBLIER_MODELE_FINAL and decay_est_fini:
    os.environ["HF_HUB_DISABLE_XET"] = "1"
    try:
        import huggingface_hub.constants as _hf_const
        import huggingface_hub._commit_api as _hf_commit
        _hf_const.HF_HUB_DISABLE_XET = True
        if hasattr(_hf_commit, "is_xet_available"):
            _hf_commit.is_xet_available = lambda: False
    except Exception:
        pass

    print(f"Phase DECAY terminée ({step_global - step_debut_decay} steps) — Fusion propre LoRA r=85 + RMSNorm...")
    modele_fusionne = modele.merge_and_unload()
    dossier_fusion = DOSSIER_TRAVAIL / "modele_fusionne_bf16"
    dossier_fusion.mkdir(parents=True, exist_ok=True)
    modele_fusionne.save_pretrained(str(dossier_fusion), safe_serialization=True)
    tokenizer.save_pretrained(str(dossier_fusion))

    client_hf.create_repo(repo_id=DEPOT_MODELE_FUSIONNE, repo_type="model", private=True, exist_ok=True)
    client_hf.upload_folder(
        folder_path=str(dossier_fusion),
        repo_id=DEPOT_MODELE_FUSIONNE,
        repo_type="model",
        commit_message=f"feat: modèle Llama 3.2 1B CPT fusionné post-decay (step {step_global}, {tokens_entraines / 1e9:.3f}B tokens, 60M params)",
    )
    print(f"Modèle CPT Llama 3.2 1B prêt pour le SFT sur : https://huggingface.co/{DEPOT_MODELE_FUSIONNE}")
elif not FUSIONNER_ET_PUBLIER_MODELE_FINAL:
    print("[INFO] Fusion désactivée (FUSIONNER_ET_PUBLIER_MODELE_FINAL=False).")
else:
    restant = max(0, PAS_DECAY_TOTAL - (step_global - (step_debut_decay or step_global)))
    print(f"[PAUSE] Phase DECAY pas encore terminée ({restant} steps restants) : relancez une session avant la fusion finale.")